# 08 Bootstrap CIs and Report Tables

Create final metric tables and bootstrap confidence intervals from score files produced by earlier notebooks. Baseline CIs are generated from notebook 06 outputs; AE CIs are generated automatically if notebook 05 score files are available.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score


In [2]:
REPO_ROOT = Path('..').resolve()
SCORE_DIR = REPO_ROOT / 'reports' / 'scores'
BASELINE_DIR = REPO_ROOT / 'reports' / 'baselines'
THRESHOLD_DIR = REPO_ROOT / 'reports' / 'thresholds'
TABLE_DIR = REPO_ROOT / 'reports' / 'tables'
FIGURE_DIR = REPO_ROOT / 'reports' / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

N_BOOTSTRAP = 1000
RANDOM_SEED = 42
AE_SCORE_COLUMNS = ['global_mse', 'global_mae', 'ver_max', 'ver_topk']


In [3]:
def evaluate_at_threshold(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'pr_auc': float(average_precision_score(y_true, scores)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }

def bootstrap_intervals(y_true: np.ndarray, scores: np.ndarray, threshold: float, n_bootstrap=N_BOOTSTRAP, seed=RANDOM_SEED) -> dict:
    rng = np.random.default_rng(seed)
    n = len(y_true)
    rows = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        sampled_y = y_true[idx]
        sampled_scores = scores[idx]
        if len(np.unique(sampled_y)) < 2:
            continue
        rows.append(evaluate_at_threshold(sampled_y, sampled_scores, threshold))
    boot = pd.DataFrame(rows)
    intervals = {}
    for metric in ['pr_auc', 'f1', 'precision', 'recall']:
        intervals[f'{metric}_ci_low'] = float(boot[metric].quantile(0.025))
        intervals[f'{metric}_ci_high'] = float(boot[metric].quantile(0.975))
    intervals['bootstrap_n_effective'] = int(len(boot))
    return intervals


In [4]:
baseline_scores_path = BASELINE_DIR / 'baseline_scores_turning.csv'
baseline_thresholds_path = THRESHOLD_DIR / 'baseline_thresholds_turning.json'
rows = []

if baseline_scores_path.exists() and baseline_thresholds_path.exists():
    baseline_scores = pd.read_csv(baseline_scores_path)
    with baseline_thresholds_path.open() as f:
        baseline_thresholds = json.load(f)
    for method, group in baseline_scores[baseline_scores['split'] == 'test'].groupby('method'):
        threshold = float(baseline_thresholds[method]['threshold'])
        y_true = group['target'].to_numpy()
        scores = group['score_value'].to_numpy()
        metrics = evaluate_at_threshold(y_true, scores, threshold)
        intervals = bootstrap_intervals(y_true, scores, threshold)
        rows.append({'dataset': 'turning', 'method': method, 'score': 'anomaly_score', 'threshold': threshold, **metrics, **intervals})

baseline_ci = pd.DataFrame(rows)
if not baseline_ci.empty:
    output_path = TABLE_DIR / 'metrics_turning_baselines_with_ci.csv'
    baseline_ci.to_csv(output_path, index=False)
    print(f'Wrote {output_path}')
baseline_ci


Wrote /home/sebastian/Dokumente/01_VSC/spectrogram-anomaly-ae/reports/tables/metrics_turning_baselines_with_ci.csv


,dataset,method,score,threshold,pr_auc,f1,precision,recall,tn,fp,...,tp,pr_auc_ci_low,pr_auc_ci_high,f1_ci_low,f1_ci_high,precision_ci_low,precision_ci_high,recall_ci_low,recall_ci_high,bootstrap_n_effective
0,turning,isolation_forest_image_features,anomaly_score,0.089329,0.989595,0.866667,1.0,0.764706,35,0,...,13,0.954784,1.0,0.705882,0.972973,1.0,1.0,0.545455,0.947368,1000
1,turning,one_class_svm_image_features,anomaly_score,0.250410,0.949249,0.903226,1.0,0.823529,35,0,...,14,0.861990,1.0,0.758405,1.000000,1.0,1.0,0.610833,1.000000,1000
2,turning,pca_image_reconstruction,anomaly_score,0.001380,0.969704,0.866667,1.0,0.764706,35,0,...,13,0.905321,1.0,0.709677,0.969740,1.0,1.0,0.550000,0.941258,1000


In [5]:
ae_score_path = SCORE_DIR / 'ae_scores_turning.csv'
ae_threshold_path = THRESHOLD_DIR / 'ae_thresholds_turning.json'
ae_rows = []

if ae_score_path.exists() and ae_threshold_path.exists():
    scores = pd.read_csv(ae_score_path)
    with ae_threshold_path.open() as f:
        thresholds = json.load(f)
    test_scores = scores[scores['split'] == 'test'].copy()
    test_scores['target'] = (test_scores['label'] == 'chatter').astype(int)
    y_true = test_scores['target'].to_numpy()
    for score_name in AE_SCORE_COLUMNS:
        threshold = float(thresholds[score_name]['threshold'])
        score_values = test_scores[score_name].to_numpy()
        metrics = evaluate_at_threshold(y_true, score_values, threshold)
        intervals = bootstrap_intervals(y_true, score_values, threshold)
        ae_rows.append({'dataset': 'turning', 'method': 'cnn_ae', 'score': score_name, 'threshold': threshold, **metrics, **intervals})
else:
    print('AE score files not found; skipping AE confidence intervals.')

ae_ci = pd.DataFrame(ae_rows)
if not ae_ci.empty:
    output_path = TABLE_DIR / 'metrics_turning_ae_with_ci.csv'
    ae_ci.to_csv(output_path, index=False)
    print(f'Wrote {output_path}')
ae_ci


Wrote /home/sebastian/Dokumente/01_VSC/spectrogram-anomaly-ae/reports/tables/metrics_turning_ae_with_ci.csv


,dataset,method,score,threshold,pr_auc,f1,precision,recall,tn,fp,...,tp,pr_auc_ci_low,pr_auc_ci_high,f1_ci_low,f1_ci_high,precision_ci_low,precision_ci_high,recall_ci_low,recall_ci_high,bootstrap_n_effective
0,turning,cnn_ae,global_mse,0.002134,0.979938,0.931034,0.870968,1.000000,44,4,...,27,0.934686,1.000000,0.836364,0.985075,0.718750,0.970588,1.000000,1.0,1000
1,turning,cnn_ae,global_mae,0.029933,0.993651,0.928571,0.896552,0.962963,45,3,...,26,0.975624,1.000000,0.842105,0.985075,0.750000,1.000000,0.869493,1.0,1000
2,turning,cnn_ae,ver_max,0.003497,0.965831,0.915254,0.843750,1.000000,43,5,...,27,0.903278,0.999131,0.816189,0.978745,0.689459,0.958375,1.000000,1.0,1000
3,turning,cnn_ae,ver_topk,0.002744,0.968217,0.915254,0.843750,1.000000,43,5,...,27,0.909413,1.000000,0.816327,0.981149,0.689655,0.962996,1.000000,1.0,1000


## Broaching Data

Add the broaching score file and threshold file here when the broaching dataset is available in the repository or mounted at a documented path.


In [7]:
# --- MA_DATASET: AE CIs ---
ma_ae_score_path = SCORE_DIR / 'ae_scores_ma_dataset.csv'
ma_ae_threshold_path = THRESHOLD_DIR / 'ae_thresholds_ma_dataset.json'
ma_ae_rows = []

if ma_ae_score_path.exists() and ma_ae_threshold_path.exists():
    ma_scores = pd.read_csv(ma_ae_score_path)
    with ma_ae_threshold_path.open() as f:
        ma_thresholds = json.load(f)

    ma_test_scores = ma_scores[ma_scores['split'] == 'test'].copy()

    # --- Label-Mapping ---
    labels_csv_path = REPO_ROOT / 'data' / '03_ma_dataset' / 'test' / 'labels.csv'
    if not labels_csv_path.exists():
        raise FileNotFoundError(f"labels.csv nicht gefunden unter {labels_csv_path}")

    labels_df = pd.read_csv(labels_csv_path)

    # Prüfe, ob 'filename' in ma_test_scores existiert
    filename_col = 'filename'
    if filename_col not in ma_test_scores.columns:
        # Versuche alternative Spaltennamen
        for col in ['image_path', 'id', 'path']:
            if col in ma_test_scores.columns:
                filename_col = col
                break
        else:
            raise KeyError(f"Keine Spalte für Dateinamen in ma_test_scores gefunden. Verfügbar: {ma_test_scores.columns.tolist()}")

    # Mappe Labels
    labels_df = labels_df.set_index('filename')
    ma_test_scores['target'] = ma_test_scores[filename_col].map(labels_df['label'])

    # Prüfe auf fehlende Labels
    if ma_test_scores['target'].isna().any():
        missing_files = ma_test_scores[ma_test_scores['target'].isna()][filename_col].unique()
        raise ValueError(f"Fehlende Dateien in labels.csv: {missing_files[:5]}...")  # Zeige nur erste 5

    y_true = ma_test_scores['target'].to_numpy()

    # --- Definiere die Score-Spalten für ma_dataset ---
    MA_AE_SCORE_COLUMNS = [
        "global_mse",
        "global_mae",  # Falls vorhanden
        "ver_max",
        "ver_topk",
        *[f"maxpool_mse_{pool_size}x{pool_size}" for pool_size in MAX_POOL_WINDOWS],
    ]
    # Filtere nur die Spalten, die tatsächlich in ma_test_scores existieren
    available_score_columns = [col for col in MA_AE_SCORE_COLUMNS if col in ma_test_scores.columns]

    for score_name in available_score_columns:
        if score_name in ma_thresholds:  # Prüfe, ob Threshold existiert
            threshold = float(ma_thresholds[score_name]['threshold'])
            score_values = ma_test_scores[score_name].to_numpy()
            metrics = evaluate_at_threshold(y_true, score_values, threshold)
            intervals = bootstrap_intervals(y_true, score_values, threshold)
            ma_ae_rows.append({
                'dataset': 'ma_dataset',
                'method': 'cnn_ae',
                'score': score_name,
                'threshold': threshold,
                **metrics,
                **intervals
            })
        else:
            print(f"Warnung: Kein Threshold für {score_name} in {ma_ae_threshold_path} gefunden.")

ma_ae_ci = pd.DataFrame(ma_ae_rows)
if not ma_ae_ci.empty:
    output_path = TABLE_DIR / 'metrics_ma_dataset_ae_with_ci.csv'
    ma_ae_ci.to_csv(output_path, index=False)
    print(f'✅ Wrote {output_path}')
else:
    print("❌ Keine AE-CIs berechnet – prüfe die Eingabedateien!")

KeyError: 'split'

In [8]:
print("Spalten in ma_scores:", ma_scores.columns.tolist())

Spalten in ma_scores: ['global_mse', 'global_mae', 'ver_max', 'ver_topk', 'maxpool_mse_3x3', 'maxpool_mse_5x5', 'maxpool_mse_9x9', 'maxpool_mse_15x15', 'maxpool_mse_25x25', 'maxpool_mse_35x35', 'maxpool_mse_51x51', 'maxpool_mse_75x75', 'maxpool_mse_101x101']
